# 2D Fokker-Planck Equation
## Advection, Diffusion, and Probability Flow

This notebook simulates the 2D Fokker-Planck equation using the pseudo-spectral `PDESolver` framework. Unlike the Schrödinger equation (which describes complex probability amplitudes), the Fokker-Planck equation describes the time evolution of the real-valued probability density function (PDF) $P(x,y,t)$ of a particle undergoing Brownian motion with drift.

---

## 1. The Governing Equation

$$
\partial_t P + \nabla \cdot (\mathbf{v} P) = D \nabla^2 P
$$

* $\mathbf{v} = (v_x, v_y)$ (Drift Velocity): Governs the deterministic advection or drift of the probability distribution.
* $D$ (Diffusion Coefficient): Governs the stochastic spreading (diffusion) of the distribution due to random fluctuations.

---

## 2. Reformulation for the Solver

We expand the divergence term (assuming constant drift) to write it as a standard evolution equation:

$$
\partial_t P = -v_x \partial_x P - v_y \partial_y P + D (\partial_x^2 P + \partial_y^2 P)
$$

In Fourier space, $\partial_x \to i\xi$ and $\partial_x^2 \to -\xi^2$, so the linear operator becomes:

$$
\text{Linear symbol:} \quad -i(v_x \xi + v_y \eta) - D(\xi^2 + \eta^2)
$$

The equation in the solver's format:

$$
\partial_t P = \underbrace{P_{\text{op}}\!\left(-i(v_x \xi + v_y \eta) - D(\xi^2 + \eta^2)\right)P} {\text{Linear drift and diffusion (Fourier space)}}
$$

---

## 3. Physical Phenomena

* **Advection (Drift)**: The center of mass of the probability distribution moves at a constant velocity $\mathbf{v}$.
* **Diffusion**: The variance of the distribution grows linearly with time ($\sigma^2 \propto 2Dt$), causing the peak to flatten.
* **Combined Dynamics**: A localized probability packet will translate across the domain while simultaneously broadening and decreasing in peak amplitude to conserve total probability.

We initialize a **localized Gaussian probability distribution** off-center to witness this advective-diffusive transport.

# Implementation
## 0. Imports 

In [ ]:
from solver import PDESolver, psiOp  # Use psiOp for real-valued fields (or psiOp if strictly required by your solver)
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters 

In [ ]:
# ── Fokker-Planck Coefficients ──
VX = 1.0      # x-drift velocity
VY = 0.5      # y-drift velocity
D  = 0.2      # Diffusion coefficient

# ── Initial Distribution Parameters ──
X0 = -4.0     # Initial x-center
Y0 = -3.0     # Initial y-center
W0 = 1.2      # Initial width (standard deviation proxy)

# ── Grid and Time ──
# Large domain to observe drift without boundary interference
Lx, Ly   = 20.0, 20.0
Nx, Ny = 512, 512    

Lt, Nt   = 30.0, 1000
n_frames = 150

## 2. Grid setup 

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol 

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
u_func    = sp.Function('u')
u_field   = u_func(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂u/∂t = -vx·∂u/∂x - vy·∂u/∂y + D·(∂²u/∂x² + ∂²u/∂y²)
# Fourier: ∂/∂x → iξ,  ∂²/∂x² → -ξ²
# So: -vx(iξ) - vy(iη) + D(-ξ² - η²) = -i(vx·ξ + vy·η) - D(ξ² + η²)

symbol_linear = -sp.I * (VX * xi + VY * eta) - D * (xi**2 + eta**2)

print("Principal symbol (linear part):")
print("  a(ξ, η) = ", symbol_linear)

## 4. Fokker-Planck equation 

In [ ]:
# ∂u/∂t = psiOp(-i(vx·ξ + vy·η) - D(ξ² + η²), u)
#        ─────────────────────────────────────
#        Linear drift and diffusion (Fourier)

equation = sp.Eq(
    sp.diff(u_field, t),
    psiOp(symbol_linear, u_field)
)

print("Fokker-Planck Equation:")
print("  ∂u/∂t = psiOp(-i(vx·ξ + vy·η) - D(ξ² + η²), u)")

## 5. Initial conditions: Localized probability packet 

In [ ]:
def initial_condition_fp(xx, yy):
    """
    Normalized 2D Gaussian probability distribution.
    Centered at (X0, Y0) with width W0.
    """
    # Normalization factor for 2D Gaussian: 1 / (pi * W0^2)
    norm = 1.0 / (np.pi * W0**2)
    
    # Gaussian profile
    profile = np.exp(-((xx - X0)**2 + (yy - Y0)**2) / W0**2)
    
    return norm * profile

## 6. Solver setup 

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',  # P -> 0 at boundaries (sufficiently large domain)
    initial_condition=initial_condition_fp,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve 

In [ ]:
frames = solver.solve()

## 8. Visualization 

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',    # Show Re(u) (probability density)
    overlay=None,  
    mode='surface',
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
# ani.save('fokker_planck_advection_diffusion.mp4', writer='ffmpeg', fps=20, dpi=100)
# print("✅ Saved to fokker_planck_advection_diffusion.mp4")